# Exploratory Data Analysis: NSE Intraday Stock Direction Predictor

> **Educational Project — Not Financial Advice.**
> This notebook explores the 5-minute OHLCV price bars, engineered scale-free features, and forward 15-minute volatility-adaptive return labels for NSE stocks.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.db import query_prices, list_tickers
from src.features import build_features, get_feature_columns
from src.labels import make_labels
from config import INDEX_TICKER, HORIZON_BARS, NEUTRAL_K

print("Available tickers in SQLite DB:", list_tickers())

## 1. Load Data for a Sample Stock

In [ ]:
sample_ticker = "ITC.NS"
df_raw = query_prices(sample_ticker)
df_raw["ts"] = pd.to_datetime(df_raw["ts"])
df = df_raw.set_index("ts").sort_index()

print(f"Loaded {len(df)} 5-minute bars for {sample_ticker}")
df.head()

## 2. Feature Distributions

In [ ]:
nifty_raw = query_prices(INDEX_TICKER)
if not nifty_raw.empty:
    nifty_raw["ts"] = pd.to_datetime(nifty_raw["ts"])
    nifty_df = nifty_raw.set_index("ts").sort_index()
else:
    nifty_df = None

feats = build_features(df.copy(), nifty_df=nifty_df, drop_nans=True)
print(f"Feature matrix shape: {feats.shape}")
print("Features computed:", get_feature_columns())

## 3. Label Distribution and Volatility-Adaptive Threshold

In [ ]:
labelled = make_labels(df.copy(), log_distribution=True)
print("Class distribution:")
print(labelled["label"].value_counts(normalize=True))

## 4. No-Lookahead Inspection
All features use strictly past or current bar information $t' \le t$. No future information is leaked.